In [50]:
import importlib
import whisky_utils
importlib.reload(whisky_utils)
from whisky_utils import *
import pandas as pd
import numpy as np
import sqlite3

conn = sqlite3.connect("whisky_auctions.db")
df_raw = pd.read_sql("""
    SELECT l.lot_id, l.title, l.winning_bid,
           l.age_years, l.distilled_date, l.bottled_date,
           l.abv, l.volume_cl, l.cask_number,
           ct.name as cask_type,
           a.auction_number
    FROM lots l
    LEFT JOIN cask_types ct ON l.cask_type_id = ct.id
    LEFT JOIN auctions a    ON l.auction_id   = a.id
""", conn)

df = enrich_dataframe(df_raw)


Applying enrichment...
Done. 115,561 lots enriched.


In [9]:
macallan = df[df["distillery"] == "Macallan"].copy()

print("Macallan regime split:")
mac_regime = (macallan.groupby(["market_regime", "is_ob"])
    .agg(
        lots=("winning_bid", "count"),
        median=("price_70cl_adj", "median")
    ).round(0))
print(mac_regime.to_string())

print("\nMacallan R1 sample — should be modern/non-sherry OB:")
mac_r1 = macallan[macallan["market_regime"] == 1].sort_values(
    "winning_bid", ascending=False)
print(mac_r1[["title", "winning_bid",
              "bottling_year_derived"]]
      .head(15).to_string(index=False))

print("\nMacallan R4 sample — should be old sherry OB + all IB:")
mac_r4 = macallan[macallan["market_regime"] == 4].sort_values(
    "winning_bid", ascending=False)
print(mac_r4[["title", "winning_bid",
              "bottling_year_derived", "is_ob"]]
      .head(15).to_string(index=False))

Macallan regime split:
                     lots  median
market_regime is_ob              
1             True   9612   179.0
4             False   599   200.0
              True    278  1520.0

Macallan R1 sample — should be modern/non-sherry OB:
                                                                         title  winning_bid  bottling_year_derived
                 Macallan 1950 Tales Of The Macallan Volume 1 Lalique Decanter      48000.0                 2021.0
                                 Macallan 72 Year Old Lalique Genesis Decanter      48000.0                    NaN
                                             Macallan 52 Year Old 2018 Release      36000.0                    NaN
                                             Macallan 52 Year Old 2018 Release      34000.0                 2018.0
                               Macallan 1950 Exceptional Cask #13 2018 Release      30000.0                 2018.0
                 Macallan Distil Your World New York, Mexico & 

In [10]:
print("Full market regime distribution:")
regime_dist = (df.groupby("market_regime")
    .agg(
        lots=("winning_bid", "count"),
        median=("price_70cl_adj", "median"),
        pct_ib=("is_ob", lambda x: (~x).mean() * 100),
        pct_closed=("is_closed", "mean")
    )
    .round(1))
print(regime_dist.to_string())

print("\nRegime 3 — full lot list (should all be holy grail):")
r3 = df[df["market_regime"] == 3].sort_values(
    "price_70cl_adj", ascending=False
)
print(r3[["title", "winning_bid", "distillery"]]
      .to_string(index=False))

print("\nRegime 1 sample — top 20 by price:")
r1 = df[df["market_regime"] == 1].sort_values(
    "price_70cl_adj", ascending=False
)
print(r1[["title", "winning_bid", "distillery"]]
      .head(20).to_string(index=False))

print("\nRegime 4 sample — top 20 by price:")
r4 = df[df["market_regime"] == 4].sort_values(
    "price_70cl_adj", ascending=False
)
print(r4[["title", "winning_bid", "distillery"]]
      .head(20).to_string(index=False))

Full market regime distribution:
                lots  median  pct_ib  pct_closed
market_regime                                   
1              99280    95.0     0.0         0.0
3                530  1289.6    15.1         0.7
4              15751   110.0    78.2         0.1

Regime 3 — full lot list (should all be holy grail):
                                                                      title  winning_bid distillery
Karuizawa 1999 & 2000 ‘The Golden House Of Five Mistress’ Mini Set 5 x 18cl       2400.0  Karuizawa
                        Port Ellen 1979 37 Year Old Goren's Whisky Mini 4cl        100.0 Port Ellen
                        Port Ellen 1979 37 Year Old Goren's Whisky Mini 4cl        100.0 Port Ellen
                                Karuizawa 1964 48 Year Old Wealth Solutions      20000.0  Karuizawa
                        Port Ellen 1979 37 Year Old Goren’s Whisky Mini 4cl         55.0 Port Ellen
                        Port Ellen 1979 37 Year Old Goren’s Whisky M

In [11]:
# Check current regime for key closed distilleries
for dist in ["Rosebank", "Brora", "Chichibu", "Dalmore"]:
    subset = df[df["distillery"] == dist]
    print(f"\n{dist} regime split:")
    print(subset.groupby("market_regime").agg(
        lots=("winning_bid", "count"),
        median=("price_70cl_adj", "median")
    ).round(0).to_string())


Rosebank regime split:
               lots  median
market_regime              
1               195   540.0
4               100   420.0

Brora regime split:
               lots  median
market_regime              
1               107  1300.0
4                64  1000.0

Chichibu regime split:
               lots  median
market_regime              
1               719   270.0
4                26   360.0

Dalmore regime split:
               lots  median
market_regime              
1               812   140.0
4                47   210.0


In [13]:
print("Full market regime distribution:")
print(df.groupby("market_regime").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median"),
    pct_ib=("is_ob", lambda x: (~x).mean() * 100)
).round(1).to_string())

for dist in ["Rosebank", "Brora", "Chichibu",
             "Port Ellen", "Springbank", "Macallan",
             "Dalmore", "Karuizawa"]:
    subset = df[df["distillery"] == dist]
    print(f"\n{dist}:")
    print(subset.groupby("market_regime").agg(
        lots=("winning_bid", "count"),
        median=("price_70cl_adj", "median")
    ).round(0).to_string())

Full market regime distribution:
                lots  median  pct_ib
market_regime                       
1              23404   120.0     0.2
2              85963    90.0    13.4
3               1153   900.0    18.8
4               5041   220.0    12.4

Rosebank:
               lots  median
market_regime              
3               295   460.0

Brora:
               lots  median
market_regime              
3               171  1200.0

Chichibu:
               lots  median
market_regime              
4               745   280.0

Port Ellen:
               lots  median
market_regime              
3               381   850.0

Springbank:
               lots  median
market_regime              
2              7801   150.0
3                66  1520.0

Macallan:
               lots  median
market_regime              
1              9612   179.0
4               877   290.0

Dalmore:
               lots  median
market_regime              
1               858   140.0
3                 1  850

In [16]:
# Check Springbank R2 composition
spring_r2 = df[
    (df["distillery"] == "Springbank") &
    (df["market_regime"] == 2)
].sort_values("price_70cl_adj", ascending=False)

print("Springbank R2 — top 20 by price:")
print(spring_r2[["title", "winning_bid",
                  "bottling_year_derived"]]
      .head(20).to_string(index=False))

print(f"\nSpringbank R2 median: "
      f"£{spring_r2['price_70cl_adj'].median():.0f}")
print(f"Springbank R2 >£500: "
      f"{(spring_r2['price_70cl_adj'] > 500).sum()}")

Springbank R2 — top 20 by price:
                                                                        title  winning_bid  bottling_year_derived
                                     Springbank Local Barley Gift Set 4 x 5cl       1500.0                 1990.0
                 Springbank 20 Year Old Celebrating 1000 Reviews at Ralfy.com       4400.0                    NaN
                       Springbank 30 Year Old Staff Only Countdown Collection       3200.0                    NaN
                       Springbank 30 Year Old Staff Only Countdown Collection       3200.0                    NaN
Springbank 1996 27 Year Old JJ Adams 'Mouse Fight' Series #1 With Prints 50cl       2100.0                 2023.0
             Springbank 1999 20 Year Old Last Of The Century Private Bottling       2800.0                 1999.0
            Springbank 2000 21 Year Old First Of The Century Private Bottling       2800.0                 2000.0
            Springbank 26 Year Old Countdown Collection

In [17]:
print("Updated full market regime distribution:")
print(df.groupby("market_regime").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median"),
    pct_ib=("is_ob", lambda x: (~x).mean() * 100)
).round(1).to_string())

# Check distilleries that might be misclassified
for dist in ["Bowmore", "Highland Park", "Ardbeg",
             "Laphroaig", "Glenlivet", "Glenfarclas"]:
    subset = df[df["distillery"] == dist]
    print(f"\n{dist}:")
    print(subset.groupby("market_regime").agg(
        lots=("winning_bid", "count"),
        median=("price_70cl_adj", "median")
    ).round(0).to_string())

Updated full market regime distribution:
                lots  median  pct_ib
market_regime                       
1              23404   120.0     0.2
2              85638    90.0    13.4
3               1478   600.0    15.0
4               5041   220.0    12.4

Bowmore:
               lots   median
market_regime               
2              1579    200.0
3                13  10964.0
4               612    150.0

Highland Park:
               lots  median
market_regime              
2              2171    80.0

Ardbeg:
               lots  median
market_regime              
2              4020   110.0
3                70   650.0

Laphroaig:
               lots  median
market_regime              
2              1817   110.0
3                12  1545.0

Glenlivet:
               lots  median
market_regime              
1              1176    65.0
2               800    65.0

Glenfarclas:
               lots  median
market_regime              
1              1398   140.0
2              

In [19]:
print("Updated regime distribution:")
print(df.groupby("market_regime").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median"),
    pct_ib=("is_ob", lambda x: (~x).mean() * 100)
).round(1).to_string())

print("\nR3 sample — should all be holy grail:")
r3 = df[df["market_regime"] == 3].sort_values(
    "price_70cl_adj", ascending=False
)
print(r3[["title", "winning_bid", "distillery"]]
      .head(30).to_string(index=False))

print("\nR3 by distillery:")
print(r3.groupby("distillery").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median")
).sort_values("median", ascending=False)
.round(0).to_string())

Updated regime distribution:
                lots  median  pct_ib
market_regime                       
1              18805   130.0     0.2
2              89097    85.0    12.5
3               3380   540.0    18.8
4               4279   230.0    13.7

R3 sample — should all be holy grail:
                                                                      title  winning_bid distillery
        Port Ellen 1983 40 Year Old Duncan Taylor Duty Paid Sample Mini 2cl        160.0 Port Ellen
              Macallan 1950 Tales Of The Macallan Volume 1 Lalique Decanter      48000.0   Macallan
Karuizawa 1999 & 2000 ‘The Golden House Of Five Mistress’ Mini Set 5 x 18cl       2400.0  Karuizawa
                        Port Ellen 1979 37 Year Old Goren's Whisky Mini 4cl        100.0 Port Ellen
                        Port Ellen 1979 37 Year Old Goren's Whisky Mini 4cl        100.0 Port Ellen
                            Macallan 1950 Exceptional Cask #13 2018 Release      30000.0   Macallan
          

In [21]:
print("Regime distribution:")
print(df.groupby("market_regime").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median")
).round(0).to_string())

print("\nR3 by distillery:")
r3 = df[df["market_regime"] == 3]
print(r3.groupby("distillery").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median")
).sort_values("median", ascending=False)
.round(0).to_string())

# Spot check key distilleries
for dist, threshold in [
    ("Ardbeg", 1981), ("Laphroaig", 1985),
    ("Glenfarclas", 1980), ("Highland Park", 1979)
]:
    subset = df[df["distillery"] == dist]
    print(f"\n{dist} (threshold {threshold}):")
    print(subset.groupby("market_regime").agg(
        lots=("winning_bid", "count"),
        median=("price_70cl_adj", "median")
    ).round(0).to_string())

Regime distribution:
                lots  median
market_regime               
1              18818   130.0
2              90084    85.0
3               2230   748.0
4               4429   240.0

R3 by distillery:
               lots  median
distillery                 
Hanyu            58  3400.0
Bowmore          52  1994.0
Karuizawa        86  1657.0
Longmorn         26  1500.0
Longrow          10  1398.0
Talisker         26  1275.0
Mortlach         26  1273.0
Brora           171  1200.0
Glenfarclas     199  1200.0
Clynelish        35   997.0
Tobermory         5   950.0
Linkwood         17   947.0
Glenlivet        35   850.0
Port Ellen      381   850.0
Laphroaig        28   849.0
Bruichladdich    12   731.0
Strathisla       21   700.0
Ardbeg           75   650.0
Highland Park    64   650.0
Glen Grant       59   650.0
Ben Nevis        20   584.0
Auchentoshan     17   560.0
Glenkinchie       1   558.0
Rosebank        295   460.0
Caol Ila         16   459.0
Cragganmore       8   305.0
Bu

In [22]:
spring_r3 = df[
    (df["distillery"] == "Springbank") &
    (df["market_regime"] == 3)
].sort_values("price_70cl_adj", ascending=False)

print(f"Springbank R3: {len(spring_r3)} lots")
print(f"Median: £{spring_r3['price_70cl_adj'].median():.0f}")
print(f">£500: {(spring_r3['price_70cl_adj'] > 500).sum()}")
print(f"<£200: {(spring_r3['price_70cl_adj'] < 200).sum()}")

print("\nSpringbank R3 — cheapest 20 (diagnosing false positives):")
print(spring_r3.tail(20)[
    ["title", "winning_bid", "price_70cl_adj",
     "bottling_year_derived"]
].to_string(index=False))

Springbank R3: 371 lots
Median: £85
>£500: 70
<£200: 280

Springbank R3 — cheapest 20 (diagnosing false positives):
                           title  winning_bid  price_70cl_adj  bottling_year_derived
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof         45.0            45.0                 2025.0
Springbank 5 Year Old 100º Proof  

In [24]:
import re
test = "Springbank 5 Year Old 100º Proof"
matches = re.findall(r'\b(19[0-9]{2})\b', test.lower())
print(f"Year matches in title: {matches}")

Year matches in title: []


In [26]:
spring_r3 = df[
    (df["distillery"] == "Springbank") &
    (df["market_regime"] == 3)
]
print(f"Springbank R3: {len(spring_r3)} lots")
print(f"Median: £{spring_r3['price_70cl_adj'].median():.0f}")
print(f">£500: {(spring_r3['price_70cl_adj'] > 500).sum()}")
print(f"<£200: {(spring_r3['price_70cl_adj'] < 200).sum()}")

print("\nSpringbank R3 cheapest 10:")
print(spring_r3.sort_values("price_70cl_adj")
      .head(10)[["title", "winning_bid",
                  "bottling_year_derived"]]
      .to_string(index=False))

Springbank R3: 96 lots
Median: £1075
>£500: 70
<£200: 1

Springbank R3 cheapest 10:
                                                           title  winning_bid  bottling_year_derived
                  Springbank 1989 15 Year Old Glenkeir Treasures        160.0                 2005.0
                            Springbank 1989 12 Year Old Rum Wood        200.0                 2002.0
SMWS 9A Glen Grant 1981 & 27 Springbank 1979 Gift Set 2 x 17.5cl        200.0                 1992.0
                            Springbank 1989 12 Year Old Rum Wood        210.0                 2002.0
                         Springbank 8 Year Old 75cl Circa 1980’s        220.0                    NaN
                           Springbank 1989 14 Year Old Port Wood        230.0                 2004.0
                           Springbank 1989 14 Year Old Port Wood        230.0                 2004.0
                       Springbank 1989 14 Year Old Old Malt Cask        230.0                 2003.0
       

In [28]:
print("Updated regime distribution:")
print(df.groupby("market_regime").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median"),
    pct_ib=("is_ob", lambda x: (~x).mean() * 100)
).round(1).to_string())

print("\nBowmore R3 cheapest 15:")
bowmore_r3 = df[
    (df["distillery"] == "Bowmore") &
    (df["market_regime"] == 3)
].sort_values("price_70cl_adj")
print(f"Total: {len(bowmore_r3)}, "
      f"Median: £{bowmore_r3['price_70cl_adj'].median():.0f}")
print(bowmore_r3.head(15)[
    ["title", "winning_bid", "bottling_year_derived"]
].to_string(index=False))

Updated regime distribution:
                lots  median  pct_ib
market_regime                       
1              18818   130.0     0.2
2              90359    85.0    12.5
3               1955   850.0    23.0
4               4429   240.0    14.1

Bowmore R3 cheapest 15:
Total: 52, Median: £1994
                                                  title  winning_bid  bottling_year_derived
       Bowmore Sherriff’s Mini 1 2/3 Fl Oz Circa 1970’s         50.0                    NaN
               Bowmore Sherriff's Mini 5cl Circa 1970's         55.0                    NaN
       Bowmore Sherriff's Mini 1 2/3 Fl Oz Circa 1970's         55.0                    NaN
         Bowmore 1972 16 Year Old Prestonfield Mini 5cl         60.0                 1988.0
                               Bowmore 1973 21 Year Old        600.0                 1994.0
Bowmore 1973 25 Year Old Blackadder Cask #3174 Mini 5cl         75.0                 1998.0
Bowmore 1973 25 Year Old Blackadder Cask #3174 Mini 5cl

In [29]:
# Check R2 lots >£1000 that might be misclassified R3
r2_expensive = df[
    (df["market_regime"] == 2) &
    (df["price_70cl_adj"] > 1000)
].sort_values("price_70cl_adj", ascending=False)

print(f"R2 lots >£1000: {len(r2_expensive)}")
print(r2_expensive[["title", "winning_bid",
                     "distillery", "price_70cl_adj"]]
      .head(30).to_string(index=False))

R2 lots >£1000: 1038
                                                                                      title  winning_bid    distillery  price_70cl_adj
                                              Acadamie Les Whiskies Francais Minis 24 x 2cl         70.0           NaN    85750.000000
                                              Kilchoman Christmas 2021 Tasting Pack 4 x 2cl         35.0     Kilchoman    42875.000000
                                  Midleton Very Rare 50 Year Old Single Cask #9158 Mini 5cl       1500.0      Midleton    12833.333333
                                                   Springbank Local Barley Gift Set 4 x 5cl       1500.0    Springbank    12833.333333
                    Glenlivet 80 Year Old Gordon & MacPhail Generations Sample Set Mini 3cl       1000.0     Glenlivet    12222.222222
                      Amrut 10 Year Old Chairman's Reserve 'Greedy Angels' With Cask Sample        900.0         Amrut     7700.000000
                                  

In [30]:
r2_expensive_70cl = df[
    (df["market_regime"] == 2) &
    (df["winning_bid"] > 1000) &
    (df["volume_cl"] == 70)
].sort_values("winning_bid", ascending=False)

print(f"R2 lots >£1000 winning bid, 70cl only: "
      f"{len(r2_expensive_70cl)}")
print(r2_expensive_70cl[
    ["title", "winning_bid", "distillery"]
].head(30).to_string(index=False))

R2 lots >£1000 winning bid, 70cl only: 719
                                                                             title  winning_bid    distillery
                            Glenfarclas 1990 30 Year Old Worlds Series #1 6 x 70cl       7000.0   Glenfarclas
                                                                    Shirakawa 1958       6000.0     Shirakawa
                                                                    Shirakawa 1958       5600.0     Shirakawa
                                                             Compass Box Scot-Free       5600.0           NaN
                              Dallas Dhu 1969 Gordon & MacPhail Private Collection       5200.0    Dallas Dhu
                                                    Glenury Royal 1953 50 Year Old       5200.0 Glenury Royal
               Littlemill 47 Year Old The Vanguards Collection No.2 Jane MacGregor       4800.0    Littlemill
                                            GlenDronach 1968 44 Year Old Rech

In [31]:
test_title = "GlenDronach 1968 44 Year Old Recherche"
import re
dist_year_match = re.search(r'\b(19[0-9]{2})\b',
                             test_title.lower())
print(f"Year match: {dist_year_match}")
if dist_year_match:
    print(f"Year found: {dist_year_match.group(1)}")
    print(f"Glendronach in era dict: "
          f"{'Glendronach' in DISTILLERY_R3_ERA}")
    print(f"GlenDronach in era dict: "
          f"{'GlenDronach' in DISTILLERY_R3_ERA}")

Year match: <re.Match object; span=(12, 16), match='1968'>
Year found: 1968
Glendronach in era dict: False
GlenDronach in era dict: False


In [32]:
print(f"Shirakawa in era dict: "
      f"{'Shirakawa' in DISTILLERY_R3_ERA}")
print(f"Shirakawa in CLOSED_DISTILLERIES: "
      f"{'Shirakawa' in CLOSED_DISTILLERIES}")

Shirakawa in era dict: False
Shirakawa in CLOSED_DISTILLERIES: True


In [41]:
print("Updated regime distribution:")
print(df.groupby("market_regime").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median")
).round(0).to_string())

# Recheck previously misclassified
for dist in ["GlenDronach", "Shirakawa", "Dallas Dhu",
             "Glenury Royal", "Littlemill"]:
    subset = df[df["distillery"] == dist]
    if len(subset) == 0:
        continue
    print(f"\n{dist}:")
    print(subset.groupby("market_regime").agg(
        lots=("winning_bid", "count"),
        median=("price_70cl_adj", "median")
    ).round(0).to_string())

# New R2 expensive check
r2_exp = df[
    (df["market_regime"] == 2) &
    (df["winning_bid"] > 1000) &
    (df["volume_cl"] == 70)
].sort_values("winning_bid", ascending=False)
print(f"\nR2 >£1000 70cl lots remaining: {len(r2_exp)}")
print(r2_exp[["title", "winning_bid", "distillery"]]
      .head(20).to_string(index=False))

Updated regime distribution:
                lots  median
market_regime               
1              19258   130.0
2              88936    85.0
3               2938   642.0
4               4429   240.0

Shirakawa:
               lots  median
market_regime              
2                 1    71.0
3                 2  5800.0

Dallas Dhu:
               lots  median
market_regime              
2                 9   170.0
3                78   290.0

Glenury Royal:
               lots  median
market_regime              
1                13   170.0
2                 3   171.0
3                49   540.0

Littlemill:
               lots  median
market_regime              
2               107   210.0
3                45   360.0

R2 >£1000 70cl lots remaining: 577
                                                                             title  winning_bid  distillery
                            Glenfarclas 1990 30 Year Old Worlds Series #1 6 x 70cl       7000.0 Glenfarclas
               

In [42]:
# Is Littlemill recognised as closed?
test = df[df["title"].str.contains("Littlemill 47", na=False)].iloc[0]
print(f"is_closed: {test['is_closed']}")
print(f"age_years: {test['age_years']}")
print(f"market_regime: {test['market_regime']}")


is_closed: True
age_years: 47.0
market_regime: 3


In [37]:
gu_r1 = df[
    (df["distillery"] == "Glenury Royal") &
    (df["market_regime"] == 1)
]
print(gu_r1[["title", "winning_bid", "is_closed",
              "is_ob"]].head(10).to_string())

                                                           title  winning_bid  is_closed  is_ob
9708        Johnnie Walker Blue Label Ghost & Rare Glenury Royal        140.0       True   True
15821       Johnnie Walker Blue Label Ghost & Rare Glenury Royal        170.0       True   True
21552       Johnnie Walker Blue Label Ghost & Rare Glenury Royal        190.0       True   True
35816       Johnnie Walker Blue Label Ghost & Rare Glenury Royal        150.0       True   True
39263       Johnnie Walker Blue Label Ghost & Rare Glenury Royal        140.0       True   True
51659       Johnnie Walker Blue Label Ghost & Rare Glenury Royal        150.0       True   True
61854  Johnnie Walker Blue Label Ghost & Rare Glenury Royal 75cl        150.0       True   True
71756  Johnnie Walker Blue Label Ghost & Rare Glenury Royal 75cl        210.0       True   True
80995    Johnnie Walker Blue Label Ghost & Rare Glenury Royal 1L        250.0       True   True
81766    Johnnie Walker Blue Label Ghost

In [43]:
laph_40 = df[df["title"].str.contains(
    "Laphroaig 40 Year Old", na=False
)].iloc[0]
print(f"age_years: {laph_40['age_years']}")
print(f"bottling_year: {laph_40['bottling_year_derived']}")
print(f"regime: {laph_40['market_regime']}")

age_years: 40.0
bottling_year: 2000.0
regime: 3


In [51]:
print("Final regime distribution:")
print(df.groupby("market_regime").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median"),
    pct_ib=("is_ob", lambda x: (~x).mean() * 100)
).round(1).to_string())

print("\nR3 by distillery (top 20):")
r3 = df[df["market_regime"] == 3]
print(r3.groupby("distillery").agg(
    lots=("winning_bid", "count"),
    median=("price_70cl_adj", "median")
).sort_values("median", ascending=False)
.head(20).round(0).to_string())

print("\nR2 >£1000 70cl remaining:")
r2_exp = df[
    (df["market_regime"] == 2) &
    (df["winning_bid"] > 1000) &
    (df["volume_cl"] == 70)
].sort_values("winning_bid", ascending=False)
print(f"Count: {len(r2_exp)}")
print(r2_exp[["title", "winning_bid", "distillery"]]
      .head(20).to_string(index=False))

Final regime distribution:
                lots  median  pct_ib
market_regime                       
1              19603   130.0     0.2
2              88324    85.0    12.3
3               3205   600.0    26.6
4               4429   240.0    14.1

R3 by distillery (top 20):
             lots  median
distillery               
Shirakawa       2  5800.0
Hanyu          58  3400.0
Bowmore        53  1994.0
Fettercairn     5  1700.0
Karuizawa      86  1657.0
Longmorn       27  1500.0
Glendronach    67  1400.0
Talisker       53  1400.0
Longrow        10  1398.0
Lagavulin      24  1350.0
Mortlach       26  1273.0
Springbank    101  1200.0
Brora         171  1200.0
Glenfarclas   202  1200.0
Killyloch       1  1050.0
Ben Wyvis       5  1027.0
Clynelish      39  1000.0
Tobermory       5   950.0
Linkwood       17   947.0
Caroni          5   900.0

R2 >£1000 70cl remaining:
Count: 519
                                                                             title  winning_bid  distillery
     

In [45]:
gf50 = df[df["title"].str.contains(
    "Glenfarclas 50 Year Old", na=False
)].iloc[0]
print(f"age: {gf50['age_years']}, "
      f"bottling_year: {gf50['bottling_year_derived']}, "
      f"regime: {gf50['market_regime']}")

age: 50.0, bottling_year: nan, regime: 2


In [48]:
test = "BOWMORE 38 YEAR OLD FOUR GUARDIAN COLLECTION"
print("four guardians" in test.lower())
print("four guardian" in test.lower())

False
True


In [52]:
laph_36 = df[df["title"].str.contains(
    "Laphroaig 36 Year Old", na=False
)].iloc[0]
print(f"age: {laph_36['age_years']}, "
      f"bottling: {laph_36['bottling_year_derived']}, "
      f"regime: {laph_36['market_regime']}")

age: 36.0, bottling: 2023.0, regime: 2


In [53]:
model_df3 = df[
    (df["winning_bid"].notna()) &
    (df["age_years"].notna()) &
    (df["age_years"] <= 100) &
    (df["abv"].notna()) &
    (df["volume_cl"].notna()) &
    (df["volume_cl"] >= 45) &
    (df["volume_cl"] <= 76) &
    (df["distillery"].notna())
].copy()

model_df3["log_price"] = np.log(model_df3["price_70cl_adj"])
model_df3["auction_recency"] = 177 - model_df3["auction_number"]
model_df3["abv_bin"] = pd.cut(
    model_df3["abv"],
    bins=[0, 40, 43, 46, 50, 55, 60, 70, 100],
    labels=[0, 1, 2, 3, 4, 5, 6, 7]
).astype(float)

model_df3["bottler_tier"] = model_df3["bottler"].map(
    BOTTLER_TIER
).fillna(model_df3["is_ob"].map({True: 1, False: 2}))
model_df3["effective_tier"] = model_df3["series_tier"].fillna(
    model_df3["bottler_tier"]
)

le_dist3 = LabelEncoder()
le_cask3 = LabelEncoder()
model_df3["distillery_enc"] = le_dist3.fit_transform(
    model_df3["distillery"].fillna("unknown")
)
model_df3["cask_enc"] = le_cask3.fit_transform(
    model_df3["cask_normalised"].fillna("unknown")
)
model_df3["is_closed"] = model_df3["is_closed"].astype(int)

features_v5 = [
    "age_years", "abv_bin", "volume_cl",
    "distillery_enc", "cask_enc", "is_closed",
    "effective_tier", "auction_recency",
    "market_regime",
]

X3 = model_df3[features_v5].copy()
y3 = model_df3["log_price"]

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42
)

model3 = xgb.XGBRegressor(
    n_estimators=500, max_depth=6,
    learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, min_child_weight=5,
    random_state=42, verbosity=0
)
model3.fit(X3_train, y3_train,
           eval_set=[(X3_test, y3_test)], verbose=False)

pred3 = model3.predict(X3_test)
r2_3  = r2_score(y3_test, pred3)
mae_3 = mean_absolute_error(np.exp(y3_test), np.exp(pred3))

print(f"Model v5 (+ market regime):")
print(f"  R²:  {r2_3:.3f}")
print(f"  MAE: £{mae_3:,.0f}")

print(f"\nFeature importance:")
importance = pd.Series(
    model3.feature_importances_,
    index=features_v5
).sort_values(ascending=False)
for feat, imp in importance.items():
    print(f"  {feat:20s}  {imp:.4f}")

NameError: name 'LabelEncoder' is not defined